# MT Benchmark — Arabic → English (MBART)

Evaluates `facebook/mbart-large-50-many-to-many-mmt` on Arabic→English translation.

**Dataset:** OPUS test set (Option A, no Drive) or UN Parallel Corpus via HuggingFace (Option B)  
**Metrics:** METEOR, BERTScore  
**Results:** See [`README.md`](README.md)

Note: during model exploration, most other Arabic→English models produced noisy output with special characters. MBART was the only model producing consistently usable translations and is the only one evaluated here.

In [ ]:
# Uncomment when running on Colab
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
%%capture
!pip install transformers sentencepiece bert_score nltk accelerate datasets
import nltk
nltk.download('wordnet')
nltk.download('wordnet_ic')
nltk.download('punkt_tab')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, AutoTokenizer
from bert_score import score as bert_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
SAMPLE_SIZE = 5_000
BATCH_SIZE  = 8       # Arabic sequences tend to be longer; reduce if OOM
MAX_LENGTH  = 512
SRC_LANG    = 'ar_AR'
TGT_LANG    = 'en_XX'
# ──────────────────────────────────────────────────────────────────────────────

## 1. Load dataset

**Option A — Tatoeba Arabic–English (no Drive):** general-domain, sentence-level, immediate download.  
**Option B — OPUS test file (from Drive):** the original evaluation data. Same format as the Tatoeba option but uses the OPUS 2019 test set.  
**Option C — UN Parallel Corpus:** `un_pc` on HuggingFace, 10K formal UN documents.

In [ ]:
# ── Option A: Tatoeba (no Drive) ─────────────────────────────────────────────
from datasets import load_dataset

raw = load_dataset('Helsinki-NLP/tatoeba_mt', 'ara-eng', split='test', trust_remote_code=True)
df = pd.DataFrame({
    'source': [r['sourceString'] for r in raw],
    'target': [r['targetString'] for r in raw],
})
df = df.dropna().sample(n=min(SAMPLE_SIZE, len(df)), random_state=1).reset_index(drop=True)
print(f'{len(df):,} sentence pairs')
df.head(3)

In [ ]:
# ── Option B: OPUS test file from Drive (uncomment to use) ───────────────────
# FILE_PATH = '/content/drive/MyDrive/YOUR_PROJECT/data/opus-2019-12-18.test.txt'
#
# from itertools import groupby
# with open(FILE_PATH, encoding='utf-8') as f:
#     lines = [l.strip() for l in f]
# parallel = [list(g) for k, g in groupby(lines, lambda x: x == '') if not k]
# df = pd.DataFrame([(p[0], p[1]) for p in parallel if len(p) >= 2], columns=['source', 'target'])
# df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=1).reset_index(drop=True)
# print(f'{len(df):,} sentence pairs')

In [ ]:
# ── Option C: UN Parallel Corpus (HuggingFace, formal domain) ────────────────
# raw = load_dataset('un_pc', 'ar-en', split='train[:10000]', trust_remote_code=True)
# df = pd.DataFrame({
#     'source': [r['translation']['ar'] for r in raw],
#     'target': [r['translation']['en'] for r in raw],
# })
# df = df.dropna().reset_index(drop=True)
# print(f'{len(df):,} sentence pairs (UN domain — expect lower METEOR: ~0.40)')

## 2. Translate — MBART

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.data[idx], return_tensors='pt',
            padding='max_length', truncation=True, max_length=self.max_length,
        )
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze()}

In [ ]:
model_name = 'facebook/mbart-large-50-many-to-many-mmt'
tokenizer  = MBart50TokenizerFast.from_pretrained(model_name)
model      = MBartForConditionalGeneration.from_pretrained(model_name)
tokenizer.src_lang = SRC_LANG
model.to(DEVICE).eval()
print(f'Model loaded on {DEVICE}')

In [ ]:
dataset    = TranslationDataset(list(df['source']), tokenizer, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

translations = []
for batch in tqdm(dataloader):
    data = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        tokens = model.generate(**data, forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG])
    translations.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))

df['translation'] = translations
print('Done.')

## 3. METEOR

In [ ]:
meteor_fn = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(df['target'], df['translation']), total=len(df)):
    scores.append(meteor_fn([ref.split()], hyp.split()))
df['meteor'] = scores
print(f'Average METEOR: {np.mean(scores):.4f}')

## 4. BERTScore

In [ ]:
tok   = AutoTokenizer.from_pretrained('roberta-large')
refs  = [' '.join(tok.tokenize(r)) for r in df['target']]
cands = [' '.join(tok.tokenize(h)) for h in df['translation']]

_, _, F1 = bert_score(cands, refs, lang='en', model_type='roberta-large', verbose=True)
df['bertscore'] = F1.numpy()
print(f'Average BERTScore F1: {F1.mean():.4f}')

## 5. Results

In [ ]:
print(df[['meteor', 'bertscore']].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label in zip(axes, ['meteor', 'bertscore'], ['METEOR', 'BERTScore']):
    ax.hist(df[col], bins=40, color='#1B1A18', alpha=0.8)
    ax.axvline(df[col].mean(), color='#C8A876', linestyle='--', label=f'mean={df[col].mean():.3f}')
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('Reference results from original experiment:')
print('  OPUS test set (5K pairs):    METEOR=0.678  BERTScore=0.964')
print('  UN parallel corpus (10K):    METEOR=0.405  BERTScore=0.916')
print()
print('Lower METEOR on UN corpus is expected — specialised terminology and formal register.')

## 6. Error analysis

In [ ]:
# Score distribution comparison
results = df[['bertscore', 'meteor']].describe().round(4)
print(results)

In [ ]:
# Sample: low BERTScore
low = df[df['bertscore'] < 0.9].sample(n=min(15, len(df[df['bertscore'] < 0.9])), random_state=1)

for _, row in low.iterrows():
    print(f'SOURCE:      {row["source"]}')
    print(f'GOLD:        {row["target"]}')
    print(f'TRANSLATION: {row["translation"]}')
    print(f'METEOR={row["meteor"]:.3f}  BERTScore={row["bertscore"]:.3f}')
    print()

In [ ]:
# Sample: high BERTScore, low METEOR — paraphrase cases
paraphrase = df[(df['bertscore'] > 0.95) & (df['meteor'] < 0.5)]
paraphrase = paraphrase.sample(n=min(10, len(paraphrase)), random_state=1)

for _, row in paraphrase.iterrows():
    print(f'SOURCE:      {row["source"]}')
    print(f'GOLD:        {row["target"]}')
    print(f'TRANSLATION: {row["translation"]}')
    print(f'METEOR={row["meteor"]:.3f}  BERTScore={row["bertscore"]:.3f}')
    print()